# 🏋️ Download & Integrate `hammamwahab/fitness-qa` Dataset

This notebook downloads the **hammamwahab/fitness-qa** dataset from Hugging Face and converts it into JSONL files compatible with the FitVoice RAG system.

**Dataset Info:**
- 123,153 rows with columns: `context`, `question`, `answer`
- Sourced from Wikipedia fitness articles via txtai
- Many rows have `"I don't have data on that"` as answer (irrelevant) — we filter those out

**Output Files:**
- `be/training_data/fitness_data.jsonl` — Q&A pairs for the RAG Q&A collection
- `be/fitness_knowledge_base_hf.jsonl` — Unique context passages as KB documents

## 1. Install Dependencies

In [1]:
!pip install datasets huggingface_hub


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Download the Dataset from Hugging Face

In [2]:
from datasets import load_dataset

print("⬇️ Downloading hammamwahab/fitness-qa from Hugging Face...")
dataset = load_dataset("hammamwahab/fitness-qa", split="train")

print(f"\n✅ Downloaded {len(dataset)} rows")
print(f"Columns: {dataset.column_names}")
print(f"\n📋 Sample row:")
print(dataset[0])

c:\Users\W1tcher\Documents\NLP_assignments\FitVoice\be\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⬇️ Downloading hammamwahab/fitness-qa from Hugging Face...

✅ Downloaded 123153 rows
Columns: ['context', 'question', 'answer']

📋 Sample row:
{'context': 'Physical fitness is a state of health and well-being and, more specifically, the ability to perform aspects of sports, occupations and daily activities. Physical fitness is generally achieved through proper nutrition, moderate-vigorous physical exercise, and sufficient rest along with a formal recovery plan.', 'question': 'Tell me about Physical fitness', 'answer': 'Physical fitness is a state of health and well-being and, more specifically, the ability to perform aspects of sports, occupations and daily activities'}


## 3. Explore & Understand the Data

In [3]:
# Check how many rows have real answers vs "I don't have data on that"
total = len(dataset)
irrelevant_count = sum(1 for row in dataset if row['answer'] == "I don't have data on that")
relevant_count = total - irrelevant_count

print(f"📊 Dataset Breakdown:")
print(f"   Total rows:      {total:,}")
print(f"   Relevant (✅):   {relevant_count:,} ({relevant_count/total*100:.1f}%)")
print(f"   Irrelevant (❌): {irrelevant_count:,} ({irrelevant_count/total*100:.1f}%)")

# Show some relevant examples
print(f"\n📋 Sample relevant Q&A pairs:")
count = 0
for row in dataset:
    if row['answer'] != "I don't have data on that":
        print(f"\n  Q: {row['question']}")
        print(f"  A: {row['answer'][:150]}..." if len(row['answer']) > 150 else f"  A: {row['answer']}")
        count += 1
        if count >= 5:
            break

📊 Dataset Breakdown:
   Total rows:      123,153
   Relevant (✅):   19,743 (16.0%)
   Irrelevant (❌): 103,410 (84.0%)

📋 Sample relevant Q&A pairs:

  Q: Tell me about Physical fitness
  A: Physical fitness is a state of health and well-being and, more specifically, the ability to perform aspects of sports, occupations and daily activitie...

  Q: Provide a quick summary on Aerobics
  A: Aerobics is a form of physical exercise that combines rhythmic aerobic exercise with stretching and strength training routines with the goal of improv...

  Q: Give an explanation on Survival of the fittest
  A: The biological concept of fitness is defined as reproductive success. In Darwinian terms, the phrase is best understood as "[s]urvival of the form tha...

  Q: Tell me about Training
  A: Training is teaching, or developing in oneself or others, any skills and knowledge or fitness that relate to specific useful competencies

  Q: Tell me about Fitbit
  A: It produces wireless-enabled wearable t

## 4. Filter & Process the Data

In [4]:
import json
import re
import os
from collections import Counter

# ============================
# Step 4a: Filter relevant Q&A pairs
# ============================
qa_pairs = []
seen_questions = set()

for row in dataset:
    # Skip irrelevant answers
    if row['answer'].strip() == "I don't have data on that":
        continue
    
    # Skip empty or very short answers
    if len(row['answer'].strip()) < 10:
        continue
    
    # Deduplicate by question text
    q_key = row['question'].strip().lower()
    if q_key in seen_questions:
        continue
    seen_questions.add(q_key)
    
    qa_pairs.append({
        "question": row['question'].strip(),
        "answer": row['answer'].strip()
    })

print(f"✅ Filtered Q&A pairs: {len(qa_pairs):,} (from {total:,} rows)")

# ============================
# Step 4b: Extract unique contexts as KB documents
# ============================
seen_contexts = set()
kb_docs = []

# Simple category classifier based on keywords
def classify_context(text):
    text_lower = text.lower()
    if any(w in text_lower for w in ['protein', 'carb', 'fat', 'calori', 'diet', 'nutrition', 'food', 'eat', 'meal', 'vitamin', 'mineral']):
        return 'nutrition'
    elif any(w in text_lower for w in ['exercise', 'workout', 'training', 'squat', 'deadlift', 'bench', 'push-up', 'pull-up', 'lift', 'rep', 'set', 'resistance', 'strength', 'cardio', 'aerobic', 'hiit']):
        return 'exercise'
    elif any(w in text_lower for w in ['muscle', 'body', 'weight', 'mass', 'lean', 'build', 'hypertrophy', 'bodybuilding']):
        return 'muscle_building'
    elif any(w in text_lower for w in ['run', 'jog', 'sprint', 'walk', 'swim', 'cycling', 'endurance', 'marathon']):
        return 'cardio'
    elif any(w in text_lower for w in ['stretch', 'yoga', 'flexibility', 'mobility', 'pilates', 'barre']):
        return 'flexibility'
    elif any(w in text_lower for w in ['injury', 'pain', 'recovery', 'rehab', 'therapy', 'heal']):
        return 'recovery'
    elif any(w in text_lower for w in ['sleep', 'rest', 'stress', 'mental', 'health', 'heart', 'blood', 'disease']):
        return 'health'
    elif any(w in text_lower for w in ['gym', 'fitness', 'sport', 'athlete', 'compete']):
        return 'fitness_general'
    else:
        return 'general'

# Extract a short title from context
def extract_title(text):
    # Take first sentence, limit to 80 chars
    first_sentence = text.split('.')[0].strip()
    if len(first_sentence) > 80:
        first_sentence = first_sentence[:77] + '...'
    return first_sentence

for row in dataset:
    context = row['context'].strip()
    
    # Skip very short contexts
    if len(context) < 30:
        continue
    
    # Deduplicate by context
    ctx_key = context.lower()[:200]
    if ctx_key in seen_contexts:
        continue
    seen_contexts.add(ctx_key)
    
    category = classify_context(context)
    title = extract_title(context)
    
    kb_docs.append({
        "category": category,
        "title": title,
        "content": context
    })

print(f"✅ Unique KB documents: {len(kb_docs):,}")

# Show category distribution
cat_counts = Counter(doc['category'] for doc in kb_docs)
print(f"\n📊 Category distribution:")
for cat, count in cat_counts.most_common():
    print(f"   {cat}: {count}")

✅ Filtered Q&A pairs: 19,743 (from 123,153 rows)
✅ Unique KB documents: 78,833

📊 Category distribution:
   general: 36273
   nutrition: 20448
   exercise: 9207
   muscle_building: 3931
   health: 3765
   fitness_general: 1957
   cardio: 1725
   recovery: 1186
   flexibility: 341


## 5. Save to JSONL Files

In [5]:
import os
import json

# Paths relative to project root
BE_DIR = os.path.join('..', 'be')
TRAINING_DATA_DIR = os.path.join(BE_DIR, 'training_data')

# Create training_data directory if it doesn't exist
os.makedirs(TRAINING_DATA_DIR, exist_ok=True)

# ============================
# Save Q&A pairs
# ============================
qa_path = os.path.join(TRAINING_DATA_DIR, 'fitness_data.jsonl')
with open(qa_path, 'w', encoding='utf-8') as f:
    for qa in qa_pairs:
        f.write(json.dumps(qa, ensure_ascii=False) + '\n')

print(f"✅ Saved {len(qa_pairs):,} Q&A pairs to {qa_path}")
print(f"   File size: {os.path.getsize(qa_path) / 1024:.1f} KB")

# ============================
# Save KB documents (HuggingFace sourced)
# ============================
kb_hf_path = os.path.join(BE_DIR, 'fitness_knowledge_base_hf.jsonl')
with open(kb_hf_path, 'w', encoding='utf-8') as f:
    for doc in kb_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"✅ Saved {len(kb_docs):,} KB documents to {kb_hf_path}")
print(f"   File size: {os.path.getsize(kb_hf_path) / 1024:.1f} KB")

# ============================
# Show quick preview
# ============================
print(f"\n📋 Preview of Q&A pairs:")
for qa in qa_pairs[:3]:
    print(f"  Q: {qa['question']}")
    print(f"  A: {qa['answer'][:100]}...\n")

print(f"📋 Preview of KB documents:")
for doc in kb_docs[:3]:
    print(f"  [{doc['category']}] {doc['title']}")
    print(f"  {doc['content'][:100]}...\n")

✅ Saved 19,743 Q&A pairs to ..\be\training_data\fitness_data.jsonl
   File size: 5166.1 KB
✅ Saved 78,833 KB documents to ..\be\fitness_knowledge_base_hf.jsonl
   File size: 43928.3 KB

📋 Preview of Q&A pairs:
  Q: Tell me about Physical fitness
  A: Physical fitness is a state of health and well-being and, more specifically, the ability to perform ...

  Q: Provide a quick summary on Aerobics
  A: Aerobics is a form of physical exercise that combines rhythmic aerobic exercise with stretching and ...

  Q: Give an explanation on Survival of the fittest
  A: The biological concept of fitness is defined as reproductive success. In Darwinian terms, the phrase...

📋 Preview of KB documents:
  [nutrition] Physical fitness is a state of health and well-being and, more specifically, ...
  Physical fitness is a state of health and well-being and, more specifically, the ability to perform ...

  [exercise] Exercise is intentional physical activity to enhance or maintain fitness and ...
  Exerci

## 6. Verify Files Are in the Right Place

In [ ]:
print("📂 Checking output files...\n")

files_to_check = [
    (qa_path, 'Q&A pairs (fitness_data.jsonl)'),
    (kb_hf_path, 'HF Knowledge Base (fitness_knowledge_base_hf.jsonl)'),
    (os.path.join(BE_DIR, 'fitness_knowledge_base.jsonl'), 'Original Knowledge Base'),
]

for path, label in files_to_check:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        with open(path, 'r', encoding='utf-8') as f:
            lines = sum(1 for _ in f)
        print(f"  ✅ {label}")
        print(f"     Path: {os.path.abspath(path)}")
        print(f"     Entries: {lines:,} | Size: {size:.1f} KB")
    else:
        print(f"  ❌ {label} — NOT FOUND at {path}")
    print()

print("\n🎉 Done! Your dataset files are ready.")
print("\n🔜 Next steps:")
print("   1. Update fitness_rag_system.py to load both KB files (see Cell 7)")
print("   2. Run setup_rag.py to rebuild the vector database")
print("   3. Test the RAG system with the expanded knowledge")

📂 Checking output files...

  ✅ Q&A pairs (fitness_data.jsonl)
     Path: c:\Users\W1tcher\Documents\NLP_assignments\FitVoice\be\training_data\fitness_data.jsonl
     Entries: 19,743 | Size: 5166.1 KB

  ✅ HF Knowledge Base (fitness_knowledge_base_hf.jsonl)
     Path: c:\Users\W1tcher\Documents\NLP_assignments\FitVoice\be\fitness_knowledge_base_hf.jsonl
     Entries: 78,833 | Size: 43928.3 KB

  ✅ Original Knowledge Base
     Path: c:\Users\W1tcher\Documents\NLP_assignments\FitVoice\be\fitness_knowledge_base.jsonl
     Entries: 44 | Size: 13.6 KB


🎉 Done! Your dataset files are ready.

🔜 Next steps:
   1. Update fitness_rag_system.py to load both KB files (see Cell 7)
   2. Run setup_rag.py to rebuild the vector database
   3. Test the RAG system with the expanded knowledge


: 

## 7. Quick Test — Load Into RAG System

This cell tests loading the new data into the RAG system to make sure everything works. Run this after confirming the JSONL files look good above.

In [ ]:
import sys
sys.path.insert(0, os.path.join('..', 'be'))

from fitness_rag_system import HybridFitnessRAG

# Initialize with both the original and new KB, plus the Q&A pairs
print("🔄 Initializing RAG system...")
rag = HybridFitnessRAG(
    knowledge_base_path=os.path.abspath(kb_hf_path),
    qa_pairs_path=os.path.abspath(qa_path)
)

# Load KB into Chroma
print("\n🔄 Loading knowledge base into Chroma...")
num_loaded = rag.load_knowledge_base()

# Get stats
stats = rag.get_full_stats()
print(f"\n✅ RAG System Stats:")
print(f"   KB Documents:  {stats['total_documents']:,}")
print(f"   Q&A Examples:  {stats['qa_examples']:,}")
print(f"   Embedding:     {stats['embedding_model']}")

# Test retrieval
test_queries = [
    "How much protein should I eat for muscle building?",
    "What is the best cardio exercise for weight loss?",
    "How does yoga improve flexibility?",
    "What are the benefits of strength training?",
]

print(f"\n{'='*60}")
print("🔍 RETRIEVAL TESTS")
print(f"{'='*60}")

for query in test_queries:
    print(f"\n📝 Query: \"{query}\"")
    context, docs = rag.retrieve_hybrid(query, kb_top_k=2, qa_top_k=1)
    if docs:
        for doc in docs[:2]:
            title = doc.get('title', doc.get('question', 'N/A'))[:60]
            relevance = doc.get('relevance', 0)
            print(f"   → [{relevance:.0%}] {title}")
    else:
        print("   ⚠️ No relevant documents found")

print(f"\n{'='*60}")
print("✅ RAG system is working with the new dataset!")
print(f"{'='*60}")

🔄 Initializing RAG system...
🔄 Loading embedding model...
🔄 Initializing Chroma database...
📖 Loading Q&A pairs from c:\Users\W1tcher\Documents\NLP_assignments\FitVoice\be\training_data\fitness_data.jsonl...
🔄 Embedding 19743 Q&A pairs...


Batches: 100%|██████████| 617/617 [00:14<00:00, 43.98it/s]


## 📊 Summary

| Metric | Before | After |
|--------|--------|-------|
| KB Documents | 45 (hand-written) | 45 + HF dataset contexts |
| Q&A Pairs | 0 | Filtered from 123K rows |
| Topic Coverage | 4 categories | 9+ categories |

### Files Created
- `be/training_data/fitness_data.jsonl` — Q&A pairs for RAG retrieval
- `be/fitness_knowledge_base_hf.jsonl` — Knowledge base documents from HF dataset